In [1]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image as PILImage
from IPython.display import Image
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
import wandb

from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    LearningRateMonitor,
    EarlyStopping,
)
from pytorch_lightning.callbacks import TQDMProgressBar
from pytorch_lightning.utilities import rank_zero_info

import sys
PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER")
SRC_DIR = str(PROJECT_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from multiomic_transformer.models import model_classifier_lightning
import multiomic_transformer.utils.classifier_data_formatter as data_formatter
import multiomic_transformer.utils.experiment_classifier_handler as experiment_handler
import multiomic_transformer.models.model_classifier as model_classifier
importlib.reload(model_classifier)
MultiomicTransformer = model_classifier.MultiomicTransformer

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

torch.set_float32_matmul_precision('medium')

GROUND_TRUTH_DIR = PROJECT_DIR / "data" / "ground_truth_files"
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/PROCESSED_DATA")

In [15]:
DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/")

cell_type="mESC"
sample_name="E7.5_rep1"
experiment_name=f"{cell_type}_{sample_name}_tutorial"
organism_code="mm10"

tdf = data_formatter.load_tdf(
    settings_path = DATA_DIR / "PROCESSED_DATA" / experiment_name / "settings.json"
)

# Verify that the data cache files exist. If not, this method will create them.
tdf.create_or_load_data_cache(sample_name=tdf.sample_names[0], force_recalculate=False)

mESC_E7.5_rep1_tutorial: Loaded existing settings
All required files are present.
Skipping data cache creation.


In [24]:
# The ExperimentHandler is a higher level class that handles the training and evaluation of the model.
# It takes in a TrainingDataFormatter object to handle file paths, data loading, and caching.
logging.info("Initializing ExperimentHandler...")
exp = experiment_handler.ExperimentHandler(
    training_data_formatter=tdf,
    experiment_dir=DATA_DIR / "EXPERIMENTS",
    model_num=1,
    silence_warnings=False,
)

# Full training run settings for the classifier.
exp.epochs = 20
exp.batch_size = 2
exp.max_pairs_per_batch = 8192

Initializing ExperimentHandler...


In [25]:
logging.info("Creating dataset...")
exp.create_multichrom_dataset(max_cached=100)

# Prepares the Train/Val/Test dataloaders, being careful to balance the number of 
# batches from each chromosome in each set.
logging.info("Preparing DataLoader...")
train_loader, val_loader, test_loader = exp.prepare_dataloader(
    batch_size=exp.batch_size,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)

# Creates scalers for the RNA and ATAC data based on a small sample of the training split.
logging.info("Creating scalers...")
exp.create_scalers(train_loader, max_batches=100)

exp.print_model_settings()

Creating dataset...
Preparing DataLoader...
Creating scalers...
Model settings for mESC_E7.5_rep1_tutorial:
  Epochs (epochs): 20
  Batch Size (batch_size): 2
  Gradient Accumulation Steps (grad_accum_steps): 1
  Use Gradient Checkpointing (use_grad_ckpt): True
  Model Dimension (d_model): 128
  Number of Heads (num_heads): 4
  Number of Layers (num_layers): 3
  Feed-Forward Dimension (d_ff): 512
  Kernel Size (kernel_size): 64
  Dropout (dropout): 0.1
  Bias Scale (bias_scale): 2.0
  Use Distance Bias (use_dist_bias): True
  Initial Learning Rate (initial_lr): 0.00025


In [26]:
tf_vocab_size = int(exp.dataset.tf_ids.numel())
tg_vocab_size = int(exp.dataset.tg_ids.numel())

model = MultiomicTransformer(
    d_model=exp.d_model,
    num_heads=exp.num_heads,
    num_layers=exp.num_layers,
    d_ff=exp.d_ff,
    dropout=exp.dropout,
    tf_vocab_size=tf_vocab_size,
    tg_vocab_size=tg_vocab_size,
    use_bias=exp.use_dist_bias,
    bias_scale=exp.bias_scale,
    window_pool_size=exp.kernel_size,
)

In [27]:
output_dir = exp.model_training_dir / "lightning_logs"
output_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Callbacks
# -------------------------------------------------------
callbacks = [
    ModelCheckpoint(
        dirpath=output_dir,
        filename="{epoch:02d}",
        monitor="val/loss",
        mode="min",
        save_top_k=3,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val/loss",
        mode="min",
        patience=5,
    ),
    LearningRateMonitor(logging_interval="step"),
]

In [ ]:
# -------------------------------------------------------
# W&B logger
# -------------------------------------------------------
rank_zero_info("Setting up Weights and Biases logger")
wandb_logger = None
wandb_logger_dir = exp.model_training_dir / "wandb_logs"
wandb_logger_dir.mkdir(parents=True, exist_ok=True)

wandb_logger = WandbLogger(
    project="multiomic_transformer_classifier",
    name=exp.experiment_name,
    save_dir=wandb_logger_dir,
    log_model=True,
)

wandb_logger.experiment.config.update({
    "model": "MultiomicTransformerClassifier",
    "d_model": exp.d_model,
    "num_heads": exp.num_heads,
    "num_layers": exp.num_layers,
    "d_ff": exp.d_ff,
    "dropout": exp.dropout,
    "use_dist_bias": exp.use_dist_bias,
    "bias_scale": exp.bias_scale,
    "kernel_size": exp.kernel_size,
})

# Hide metrics from auto-generated W&B charts
wandb_logger.experiment.define_metric("trainer/global_step", hidden=True)
wandb_logger.experiment.define_metric("epoch", hidden=True)
wandb_logger.experiment.define_metric("lr-AdamW", hidden=True)

rank_zero_info("Setting up PyTorch Lightning Trainer...")
trainer = pl.Trainer(
    max_epochs=exp.epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    strategy="auto",
    logger=wandb_logger,
    callbacks=[TQDMProgressBar(refresh_rate=5)] + callbacks,
    gradient_clip_val=1.0,
    deterministic=True,
    default_root_dir=output_dir,
    enable_progress_bar=True,
    enable_checkpointing=True,
    check_val_every_n_epoch=1,
    # limit_train_batches=1024,
    # limit_val_batches=256,
)

chipatlas_edge_df = exp.load_ground_truth(GROUND_TRUTH_DIR / "chip_atlas_tf_peak_tg_dist.csv")[0]

importlib.reload(model_classifier_lightning)
lit_model = model_classifier_lightning.LitMultiomicTransformer(
    exp=exp,
    model=model,
    ground_truth_edges=chipatlas_edge_df,
    ground_truth_name="ChIP-Atlas mESC",
    ground_truth_negative_ratio=10.0,
    max_ground_truth_pairs=10000
)

trainer.fit(lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
wandb.finish()

Setting up Weights and Biases logger


Setting up PyTorch Lightning Trainer...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ MultiomicTransformer   │  1.9 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss      │      0 │ train │     0 │
│ 2 │ val_auroc │ BinaryAUROC            │      0 │ train │     0 │
│ 3 │ val_auprc │ BinaryAveragePrecision │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.9 M                                                                                                
Total estimated model params size (MB): 7                                                                          
Modules in train mode: 103                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |                                                                                    | 0/? [0…

Training: |                                                                                           | 0/? [0…

In [ ]:
wandb.finish()

trainer/global_step,▁
trainer/global_step,0


epoch,▁
lr_plateau,▁
train/step_time_sec,▁
trainer/global_step,▁██
epoch,0
lr_plateau,0.00025
train/step_time_sec,0.02606
trainer/global_step,49
